# Модели бинарного выбора (пробит и логит)

In [1]:
import statsmodels.api as sm

Продемонстрируем оценивание пробит и логит моделей на примере из работы Spector & Mazzeo (1980):

* Зависимая переменная равна 0, если студент набрал менее 2,5 баллов в тесте на освоение курса макроэкономики.
* `GPA` -- средняя кумулятивная оценка по двум курсам макроэкономики,
* `TUCE` -- результаты экзамена,
* `PCI` -- участие студента в персонализированной программе обучения.

In [2]:
spector_data = sm.datasets.spector.load_pandas()
spector_data.endog, spector_data.exog

(0     0.0
 1     0.0
 2     0.0
 3     0.0
 4     1.0
 5     0.0
 6     0.0
 7     0.0
 8     0.0
 9     1.0
 10    0.0
 11    0.0
 12    0.0
 13    1.0
 14    0.0
 15    0.0
 16    0.0
 17    0.0
 18    0.0
 19    1.0
 20    0.0
 21    1.0
 22    0.0
 23    0.0
 24    1.0
 25    1.0
 26    1.0
 27    0.0
 28    1.0
 29    1.0
 30    0.0
 31    1.0
 Name: GRADE, dtype: float64,
      GPA  TUCE  PSI
 0   2.66  20.0  0.0
 1   2.89  22.0  0.0
 2   3.28  24.0  0.0
 3   2.92  12.0  0.0
 4   4.00  21.0  0.0
 5   2.86  17.0  0.0
 6   2.76  17.0  0.0
 7   2.87  21.0  0.0
 8   3.03  25.0  0.0
 9   3.92  29.0  0.0
 10  2.63  20.0  0.0
 11  3.32  23.0  0.0
 12  3.57  23.0  0.0
 13  3.26  25.0  0.0
 14  3.53  26.0  0.0
 15  2.74  19.0  0.0
 16  2.75  25.0  0.0
 17  2.83  19.0  0.0
 18  3.12  23.0  1.0
 19  3.16  25.0  1.0
 20  2.06  22.0  1.0
 21  3.62  28.0  1.0
 22  2.89  14.0  1.0
 23  3.51  26.0  1.0
 24  3.54  24.0  1.0
 25  2.83  27.0  1.0
 26  3.39  17.0  1.0
 27  2.67  24.0  1.0
 28  3.65

In [3]:
spector_data.exog = sm.add_constant(spector_data.exog)

In [4]:
spector_data.exog

,const,GPA,TUCE,PSI
0,1.0,2.66,20.0,0.0
1,1.0,2.89,22.0,0.0
2,1.0,3.28,24.0,0.0
3,1.0,2.92,12.0,0.0
4,1.0,4.00,21.0,0.0
5,1.0,2.86,17.0,0.0
6,1.0,2.76,17.0,0.0
7,1.0,2.87,21.0,0.0
8,1.0,3.03,25.0,0.0
9,1.0,3.92,29.0,0.0


## Пробит

Пробит модель кратко описывается выражением
$$\Pr(Y=1) = \Phi(x'\beta)$$

```python
class statsmodels.discrete.discrete_model.Probit(
  endog, 
  exog, 
  offset=None, 
  check_rank=True, 
  **kwargs
)
```
* `offset` --- сдвиг, добавляемый к линейной модели в $\Phi()$.
* `check_rank` --- оценка ранга `exog`. Если `false`, то модель инициализируется быстрее.
* `missing` --- `none`, `drop` или `raise`.

Оценим модель при помощи `fit`
```python
Probit.fit(
  start_params=None,
  method='newton',
  maxiter=35,
  full_output=1,
  disp=1,
  callback=None,
  **kwargs
)
```
* `start_params` --- начальные значения параметров, по-умолчанию нули.
* `method`:
  * ‘newton’ for Newton-Raphson,
  * ‘nm’ for Nelder-Mead
  * ‘bfgs’ for Broyden-Fletcher-Goldfarb-Shanno (BFGS)
  * ‘lbfgs’ for limited-memory BFGS with optional box constraints
  * ‘powell’ for modified Powell’s method
  * ‘cg’ for conjugate gradient
  * ‘ncg’ for Newton-conjugate gradient
  * ‘basinhopping’ for global basin-hopping solver
  * ‘minimize’ for generic wrapper of scipy minimize (BFGS by default)
  
  `kwargs` передаются соответствующему солверу, см. `scipy.optimize`.
* `full_output` --- выводить ли ВСЕ результаты.
* `disp` --- выводить ли сообщения о сходимости.
* `fargs` --- дополнительные аргументы для функции правдоподобия.
* `callback` --- функция, вызываемая после каждой итерации.

In [5]:
probit_mod = sm.Probit(spector_data.endog, spector_data.exog)
probit_mod_est = probit_mod.fit()

Optimization terminated successfully.
         Current function value: 0.400588
         Iterations 6


In [6]:
print(probit_mod_est.summary())

                          Probit Regression Results                           
Dep. Variable:                  GRADE   No. Observations:                   32
Model:                         Probit   Df Residuals:                       28
Method:                           MLE   Df Model:                            3
Date:                Wed, 14 Oct 2026   Pseudo R-squ.:                  0.3775
Time:                        10:54:14   Log-Likelihood:                -12.819
converged:                       True   LL-Null:                       -20.592
Covariance Type:            nonrobust   LLR p-value:                  0.001405
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -7.4523      2.542     -2.931      0.003     -12.435      -2.469
GPA            1.6258      0.694      2.343      0.019       0.266       2.986
TUCE           0.0517      0.084      0.617      0.5

Так как классическая интерпретация коэффициентов в случае пробит модели не работает ($\beta_i$ показывает изменение **скрытой** переменой в ответ на изменение $x_i$ на единицу). Для нормальной интерпретации смотрят на предельный эффект
$$\frac{\partial \Phi(x'\beta)}{\partial \beta_i} = \phi(x'\beta) \beta_i$$
Очевидно, что предельный эффект зависит от $x$, поэтому в следующей функции рассматриваются разные сценарии расчета эффектов.

Оценим предельные эффекты (влияние на вероятности)
```python
BinaryResults.get_margeff(
  at='overall',
  method='dydx', 
  atexog=None, 
  dummy=False, 
  count=False
)
```
* `at`:
  * `overall` --- средний предельный эффект (считаются предельные эффекты для разных значений $x$ и усредняются);
  * `mean` --- эффект для среднего значения $\bar{x}$;
  * `median` --- аналогично для медианы;
  * `zero` --- предельный эффект для $x=0$;
  * `all` --- предельные эффекты для каждого наблюдения. В этом случае в итоговом объекте будет доступен только атрибут `margeff`.
* `method`
  * `dydx` --- простой эффект
  * `eyex` --- расчет эластичностей
  * `dyex` и `eydx` --- полуэластичности.
* `atexog` --- опциональные значения экзогенных переменных
* `dummy` и `count` --- рассматривать бинарные и счетные переменные как непрерывные

In [7]:
margin_eff = probit_mod_est.get_margeff()
margin_eff.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
       Probit Marginal Effects       
=====================================
Dep. Variable:                  GRADE
Method:                          dydx
At:                           overall
==============================================================================
                dy/dx    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
GPA            0.3608      0.113      3.182      0.001       0.139       0.583
TUCE           0.0115      0.018      0.624      0.533      -0.025       0.048
PSI            0.3165      0.090      3.508      0.000       0.140       0.493
==============================================================================
"""

In [8]:
margin_eff = probit_mod_est.get_margeff(method="eyex")
margin_eff.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
       Probit Marginal Effects       
=====================================
Dep. Variable:                  GRADE
Method:                          eyex
At:                           overall
=================================================================================
           d(lny)/d(lnx)    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
GPA               6.3578      3.212      1.979      0.048       0.062      12.654
TUCE              1.4398      2.369      0.608      0.543      -3.203       6.083
PSI               0.4656      0.164      2.837      0.005       0.144       0.787
=================================================================================
"""

In [9]:
margin_eff = probit_mod_est.get_margeff(at="zero")
margin_eff.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
       Probit Marginal Effects       
=====================================
Dep. Variable:                  GRADE
Method:                          dydx
At:                              zero
==============================================================================
                dy/dx    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
GPA         5.653e-13   1.06e-11      0.054      0.957   -2.01e-11    2.12e-11
TUCE        1.799e-14   3.28e-13      0.055      0.956   -6.25e-13    6.61e-13
PSI         4.959e-13   9.32e-12      0.053      0.958   -1.78e-11    1.88e-11
==============================================================================
"""

In [10]:
margin_eff = probit_mod_est.get_margeff(at="mean")
margin_eff.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
       Probit Marginal Effects       
=====================================
Dep. Variable:                  GRADE
Method:                          dydx
At:                              mean
==============================================================================
                dy/dx    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
GPA            0.5333      0.232      2.294      0.022       0.078       0.989
TUCE           0.0170      0.027      0.626      0.531      -0.036       0.070
PSI            0.4679      0.188      2.494      0.013       0.100       0.836
==============================================================================
"""

In [11]:
margin_eff = probit_mod_est.get_margeff(at="all")
margin_eff.margeff.mean(axis=0)

array([0.36078629, 0.01147926, 0.31651986])

## Логит

С ним все аналогично, только вместо `Probit` надо использовать `Logit`.

In [12]:
logit_mod = sm.Logit(spector_data.endog, spector_data.exog)
logit_mod_est = logit_mod.fit()

Optimization terminated successfully.
         Current function value: 0.402801
         Iterations 7


In [13]:
print(logit_mod_est.summary())

                           Logit Regression Results                           
Dep. Variable:                  GRADE   No. Observations:                   32
Model:                          Logit   Df Residuals:                       28
Method:                           MLE   Df Model:                            3
Date:                Wed, 14 Oct 2026   Pseudo R-squ.:                  0.3740
Time:                        10:54:14   Log-Likelihood:                -12.890
converged:                       True   LL-Null:                       -20.592
Covariance Type:            nonrobust   LLR p-value:                  0.001502
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const        -13.0213      4.931     -2.641      0.008     -22.687      -3.356
GPA            2.8261      1.263      2.238      0.025       0.351       5.301
TUCE           0.0952      0.142      0.672      0.5

In [14]:
logit_mod_est.f_test("GPA=PSI=2.5")

<class 'statsmodels.stats.contrast.ContrastResults'>
<F test: F=0.05472540949205792, p=0.9468460852763116, df_denom=28, df_num=2>

In [15]:
margin_eff = logit_mod_est.get_margeff(method="dydx")
margin_eff.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
        Logit Marginal Effects       
=====================================
Dep. Variable:                  GRADE
Method:                          dydx
At:                           overall
==============================================================================
                dy/dx    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
GPA            0.3626      0.109      3.313      0.001       0.148       0.577
TUCE           0.0122      0.018      0.686      0.493      -0.023       0.047
PSI            0.3052      0.092      3.304      0.001       0.124       0.486
==============================================================================
"""

In [16]:
margin_eff = logit_mod_est.get_margeff(at="zero")
margin_eff.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
        Logit Marginal Effects       
=====================================
Dep. Variable:                  GRADE
Method:                          dydx
At:                              zero
==============================================================================
                dy/dx    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
GPA         6.253e-06   2.88e-05      0.217      0.828   -5.03e-05    6.28e-05
TUCE        2.105e-07   9.24e-07      0.228      0.820    -1.6e-06    2.02e-06
PSI         5.263e-06    2.5e-05      0.211      0.833   -4.37e-05    5.42e-05
==============================================================================
"""

In [17]:
margin_eff = logit_mod_est.get_margeff(at="all")
margin_eff.margeff

array([[0.07311607, 0.00246188, 0.06154047],
       [0.15815168, 0.0053251 , 0.1331134 ],
       [0.43011639, 0.01448239, 0.36202116],
       [0.07130492, 0.0024009 , 0.06001605],
       [0.69272252, 0.02332457, 0.58305197],
       [0.09507939, 0.00320141, 0.08002659],
       [0.0729182 , 0.00245522, 0.06137393],
       [0.1381988 , 0.00465327, 0.11631942],
       [0.2791564 , 0.00939944, 0.23496087],
       [0.60069976, 0.02022608, 0.50559808],
       [0.06746375, 0.00227156, 0.05678301],
       [0.43493424, 0.01464461, 0.36607625],
       [0.61722645, 0.02078255, 0.51950829],
       [0.44053615, 0.01483323, 0.37079128],
       [0.65191691, 0.02195061, 0.54870662],
       [0.08272793, 0.00278552, 0.0696306 ],
       [0.14342697, 0.00482931, 0.12071987],
       [0.10484673, 0.00353028, 0.08824759],
       [0.68370145, 0.02302082, 0.5754591 ],
       [0.63346725, 0.02132939, 0.53317788],
       [0.16280911, 0.00548192, 0.13703347],
       [0.24332458, 0.00819295, 0.20480188],
       [0.

## Мультиномиальный логит

Рассмотрим пример на данных `American National Election Studies`.
В целом все абсолютно аналогично пробиту и логиту...

In [18]:
anes_data = sm.datasets.anes96.load()
anes_exog = anes_data.exog
anes_exog = sm.add_constant(anes_exog)

In [19]:
anes_data.endog

0      6.0
1      1.0
2      1.0
3      1.0
4      0.0
      ... 
939    4.0
940    6.0
941    5.0
942    6.0
943    3.0
Name: PID, Length: 944, dtype: float64

In [20]:
mlogit_mod = sm.MNLogit(anes_data.endog, anes_exog)
mlogit_res = mlogit_mod.fit()
mlogit_res.params

Optimization terminated successfully.
         Current function value: 1.548647
         Iterations 7


,0,1,2,3,4,5
const,-0.373402,-2.250913,-3.665584,-7.613843,-7.060478,-12.105751
logpopul,-0.011536,-0.088751,-0.105967,-0.091557,-0.093285,-0.140881
selfLR,0.297714,0.391669,0.573451,1.278772,1.346962,2.070080
age,-0.024945,-0.022898,-0.014851,-0.008681,-0.017904,-0.009433
educ,0.082491,0.181043,-0.007152,0.199828,0.216939,0.321926
income,0.005197,0.047874,0.057575,0.084498,0.080958,0.108894


In [21]:
mlogit_res.get_margeff().summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
       MNLogit Marginal Effects      
=====================================
Dep. Variable:                    PID
Method:                          dydx
At:                           overall
==============================================================================
     PID=0      dy/dx    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
logpopul       0.0087      0.004      2.250      0.024       0.001       0.016
selfLR        -0.0978      0.008    -12.153      0.000      -0.114      -0.082
age            0.0027      0.001      3.856      0.000       0.001       0.004
educ          -0.0199      0.008     -2.420      0.016      -0.036      -0.004
income        -0.0060      0.002     -2.977      0.003      -0.010      -0.002
------------------------------------------------------------------------------
     PID=1      dy/dx    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
logpopul       0.0070      0.004      1.791      0.073      -0.001       0.015
selfLR        -0.0502      0.007     -6.824      0.000      -0.065      -0.036
age           -0.0021      0.001     -2.789      0.005      -0.004      -0.001
educ          -0.0054      0.008     -0.636      0.525      -0.022       0.011
income        -0.0055      0.002     -2.707      0.007      -0.010      -0.002
------------------------------------------------------------------------------
     PID=2      dy/dx    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
logpopul      -0.0039      0.003     -1.246      0.213      -0.010       0.002
selfLR        -0.0282      0.006     -4.972      0.000      -0.039      -0.017
age           -0.0010      0.001     -1.523      0.128      -0.002       0.000
educ           0.0066      0.007      0.964      0.335      -0.007       0.020
income         0.0010      0.002      0.530      0.596      -0.003       0.005
------------------------------------------------------------------------------
     PID=3      dy/dx    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
logpopul      -0.0018      0.002     -0.940      0.347      -0.006       0.002
selfLR        -0.0057      0.003     -1.798      0.072      -0.012       0.001
age        -4.249e-05      0.000     -0.110      0.912      -0.001       0.001
educ          -0.0055      0.004     -1.253      0.210      -0.014       0.003
income         0.0005      0.001      0.469      0.639      -0.002       0.003
------------------------------------------------------------------------------
     PID=4      dy/dx    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
logpopul      -0.0010      0.003     -0.330      0.741      -0.007       0.005
selfLR         0.0199      0.005      3.672      0.000       0.009       0.030
age            0.0005      0.001      0.815      0.415      -0.001       0.002
educ           0.0017      0.006      0.268      0.789      -0.011       0.014
income         0.0021      0.002      1.119      0.263      -0.002       0.006
------------------------------------------------------------------------------
     PID=5      dy/dx    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
logpopul      -0.0015      0.004     -0.429      0.668      -0.009       0.005
selfLR         0.0376      0.007      5.404      0.000       0.024       0.051
age           -0.0007      0.001     -0.949      0.343      -0.002       0.001
educ           0.0047      0.008      0.604      0.546      -0.011       0.020
income         0.0025      0.002      1.140      

# Цензурированные регрессии

Цензурированные регрессии --- это регрессионные модели, в которых зависиммая переменная наблюдаема не на всем своем диапазоне.
Например, если пытаться оценить модель, показывающую желаемую цену автомобила в зависимости от доходов семьи, с использованием данных по цене уже купленных автомобилей, то такие данные будут ограничены минимальной ценой автомобиля в выборке.
То есть семьи без автомобиля хотели бы купить автомобиль по цене меньше минимальной:
* Если опустить данные наблюдения, то получим смещение отбора,
* Если заменить отсутствующие $y$ на минимальное значение цены и воспользоваться МНК, то получим смещение из-за систематической ошибки в $y$.

Допустим, что нас интересует модель
$$y^*_i = x_i'\beta + \varepsilon_i$$
где $y^*$ ненаблюдаемая. Вместо нее мы имеем
$$y_i = \begin{cases}y^*_i,& y^*_i \gt 0 \\0,& y^*_i \le 0\end{cases}$$

In [22]:
import numpy as np
from py4etrics.tobit import Tobit

In [23]:
from linearmodels.datasets import mroz

print(mroz.DESCR)
data = mroz.load()

# Количество отработанных часов в неделю
data["hour"] = data["hours"] / 52

data.head()


T.A. Mroz (1987), "The Sensitivity of an Empirical Model of Married Women's
Hours of Work to Economic and Statistical Assumptions," Econometrica 55,
765-799.

nlf        1 if in labor force, 1975
hours      hours worked, 1975
kidslt6    # kids < 6 years
kidsge6    # kids 6-18
age        woman's age in yrs
educ       years of schooling
wage       estimated wage from earns., hours
repwage    reported wage at interview in 1976
hushrs     hours worked by husband, 1975
husage     husband's age
huseduc    husband's years of schooling
huswage    husband's hourly wage, 1975
faminc     family income, 1975
mtr        fed. marginal tax rate facing woman
motheduc   mother's years of schooling
fatheduc   father's years of schooling
unem       unem. rate in county of resid.
city       =1 if live in SMSA
exper      actual labor mkt exper
nwifeinc   (faminc - wage*hours)/1000
lwage      log(wage)
expersq    exper^2



,inlf,hours,kidslt6,kidsge6,age,educ,wage,repwage,hushrs,husage,...,mtr,motheduc,fatheduc,unem,city,exper,nwifeinc,lwage,expersq,hour
0,1,1610,1,0,32,12,3.3540,2.65,2708,34,...,0.7215,12,7,5.0,0,14,10.910060,1.210154,196,30.961538
1,1,1656,0,2,30,12,1.3889,2.65,2310,30,...,0.6615,7,7,11.0,1,5,19.499980,0.328512,25,31.846154
2,1,1980,1,3,35,12,4.5455,4.04,3072,40,...,0.6915,12,7,5.0,0,15,12.039910,1.514138,225,38.076923
3,1,456,0,3,34,12,1.0965,3.25,1920,53,...,0.7815,7,7,5.0,0,6,6.799996,0.092123,36,8.769231
4,1,1568,1,2,31,14,4.5918,3.60,2000,32,...,0.6215,12,14,9.5,1,7,20.100060,1.524272,49,30.153846


In [25]:
# Заполним пропуски в `lwage` нулями
data["lwage_c"] = data["lwage"]
data.loc[np.isnan(data["lwage"]), "lwage_c"] = 0

# Введём цензуру: 0 нет цензуры, -1 есть цензура низу, 1 есть цензура сверху
data["censor"] = 0
data.loc[np.isnan(data["lwage"]), "censor"] = -1

Оценим Тобит модель.
Так как пакет `py4etrics` писался под большим впечатлением от `statsmodels`, то интерфейс в целом понятен и аналогичен.

```python
Tobit(y, X, cens=<A>, left=<B>, right=<C>, data=<D>)
Tobit.from_formula(formula, cens=<A>, left=<B>, right=<C>, data=<D>)
```

* `cens`: переменная, равная -1, если данное значение $y$ ограничено снизу, 1 --- если сверху, 0 --- ессли наблюдение не ограничено;
* `left` и `right`: соответственно нижнее и верхнее ограничения.

In [26]:
X = data[
    [
        "educ",
        "exper",
        "expersq",
        "age",
        "city",
        "kidslt6",
        "kidsge6",
        "nwifeinc",
        "unem",
    ]
]
X = sm.add_constant(X)

res_tobit = Tobit(data.lwage_c, X, cens=data["censor"], left=0, data=data).fit()

print(res_tobit.summary())

Optimization terminated successfully.
         Current function value: 1.102030
         Iterations: 1789
         Function evaluations: 2490
                              Tobit Regression Results                             
Dep. Variable:                     lwage_c   Pseudo R-squ:                    0.138
Method:                 Maximum Likelihood   Log-Likelihood:                 -829.8
No. Observations:                      753   LL-Null:                        -962.3
No. Uncensored Obs:                    428   LL-Ratio:                        265.0
No. Left-censored Obs:                 325   LLR p-value:                     0.000
No. Right-censored Obs:                  0   AIC:                            1679.7
Df Residuals:                          743   BIC:                            1725.9
Df Model:                                9   Covariance Type:             nonrobust
                 coef    std err          z      P>|z|      [0.025      0.975]
-----------------------

Результат оценивания содержит практически те же методы, что и результаты оценивания МНК.

In [27]:
res_tobit.t_test("educ = 1")

<class 'statsmodels.stats.contrast.ContrastResults'>
                             Test for Constraints                             
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
c0             0.1598      0.020    -41.535      0.000       0.120       0.199